# RAG 기본 프로세스

In [65]:
!pip install -r requirements.txt

In [2]:
!ollama pull bge-m3
!ollama pull gemma4:e2b

pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest 
pulling daec91ffb5dd: 100% ▕██████████████████▏ 1.2 GB                         
pulling a406579cd136: 100% ▕██████████████████▏ 1.1 KB                         
pulling 0c4c9c2a325f: 100% ▕██████████████████▏  337 B                         
verifying sha256 digest 
writing manifest 
success 
pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest 
pulling 4e30e2665218: 100% ▕██████████████████▏ 7.2 GB                         
pulling 7339fa418c9a: 100% ▕██████████████████▏  11 KB                         
pulling 56380ca2ab89: 100% ▕██████████████████▏   42 B                         
pulling c6bc3775a3fa: 100% ▕██████████████████▏  473 B                 

### 라이브러리 설치

In [ ]:

# langchain, langchain-community, langchain-openai: RAG 구성의 핵심
# chromadb: 벡터 데이터베이스
# pypdf: PDF 로딩
# sentence-transformers: 한국어 임베딩
# openai: GPT/임베딩 연동
# llama-cpp-python, ollama: 로컬/서버형 모델
# ★ llama-cpp-python 및 ollama는 필요 시

# !pip install llama-cpp-python==0.3.16

### 라이브러리 선언

In [ ]:
import os
from dotenv import load_dotenv

import json

from opensearchpy import OpenSearch

from tqdm import tqdm

from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_community.embeddings import OllamaEmbeddings

from langchain_community.llms import Ollama

from langchain_core.prompts import PromptTemplate

from langchain_core.runnables import RunnablePassthrough, RunnableParallel

from langchain_core.output_parsers import StrOutputParser

from langchain_community.vectorstores import OpenSearchVectorSearch


### 환경 설정 *** 본인 OPENAI_API키 로 설정 필요

In [15]:
load_dotenv()

# OpenSearch 설정
OPENSEARCH_HOST = os.getenv('OPENSEARCH_HOST')
OPENSEARCH_PORT = int(os.getenv('OPENSEARCH_PORT'))
OPENSEARCH_USER = os.getenv('OPENSEARCH_USER')
OPENSEARCH_PASSWORD = os.getenv('OPENSEARCH_PASSWORD')
OPENSEARCH_INDEX = os.getenv('OPENSEARCH_INDEX')

# LLM 모델 선택
MODEL_TYPE =  os.getenv('MODEL_TYPE')
OLLAMA_MODEL = os.getenv('OLLAMA_MODEL')

# 임베딩 모델 설정
EMBED_MODEL_NAME = os.getenv('EMBED_MODEL_NAME')

# 적재 모드 설정

print("OpenSearch 호스트:", OPENSEARCH_HOST)
print("인덱스:", OPENSEARCH_INDEX)

OpenSearch 호스트: 192.168.110.111
인덱스: my_vector_index
적재 모드: full


In [16]:
# 설정값 정의
CHUNK_SIZE = 600        # 청크 크기 (권장: 600-1200)
CHUNK_OVERLAP = 100     # 청크 오버랩 (권장: 100-200)
K_RETRIEVAL = 4         # 검색할 문서 수 (권장: 3-6)

print("설정된 하이퍼파라미터:")
print(f"청크 크기: {CHUNK_SIZE}")
print(f"오버랩: {CHUNK_OVERLAP}")
print(f"검색 수: {K_RETRIEVAL}")

print("\n 설정값 영향:")
print("   - 큰 청크: 문맥 유지↑, 검색 정밀도↓")
print("   - 많은 오버랩: 문맥 단절 방지, 중복↑")
print("   - 높은 K값: 정확도↑, 속도/비용↑")

설정된 하이퍼파라미터:
청크 크기: 600
오버랩: 100
검색 수: 4

 설정값 영향:
   - 큰 청크: 문맥 유지↑, 검색 정밀도↓
   - 많은 오버랩: 문맥 단절 방지, 중복↑
   - 높은 K값: 정확도↑, 속도/비용↑


# 1. 데이터 준비

### 1-1. Data Load

In [ ]:
# JSON 파일 로딩
JSON_PATH = os.getenv('JSON_PATH')

with open(JSON_PATH, 'r', encoding='utf-8') as f:
    raw_data = json.load(f)

# 리스트면 그대로, 딕셔너리면 값 추출
if isinstance(raw_data, list):
    data = raw_data
elif isinstance(raw_data, dict):
    data = list(raw_data.values())[0]

print(f"{len(data)}개 데이터 로딩 완료!")
print(f"첫 번째 데이터 미리보기: {data[0]}")

5개 데이터 로딩 완료!
첫 번째 데이터 미리보기: {'id': 1, 'title': '갤럭시 S24 울트라', 'category': '스마트폰', 'description': '삼성 갤럭시 S24 울트라는 삼성전자가 2024년 1월에 출시한 플래그십 스마트폰으로, 200MP 메인 카메라를 포함한 쿼드 카메라 시스템을 탑재하고 있습니다. 카메라 시스템은 200MP 광각, 12MP 초광각, 50MP 3배 망원, 10MP 10배 망원으로 구성되어 있으며, AI 기반의 사진 편집 기능인 갤럭시 AI를 통해 사진 보정, 객체 제거, 배경 변경 등 다양한 편집 기능을 제공합니다. 디스플레이는 6.8인치 Dynamic AMOLED 2X 패널을 사용하며 최대 120Hz 주사율을 지원합니다. 프로세서는 Snapdragon 8 Gen 3를 탑재하여 강력한 성능을 발휘하며, 5000mAh 대용량 배터리와 45W 고속 충전을 지원합니다. S펜이 내장되어 있어 메모 작성 및 그림 그리기에 활용할 수 있으며, IP68 방수 방진 등급을 지원합니다. 티타늄 소재의 프레임을 사용하여 내구성이 뛰어나며, 다양한 색상 옵션을 제공합니다. 삼성 덱스를 지원하여 외부 모니터에 연결하면 데스크톱 환경으로 사용할 수 있으며, 삼성 페이를 통한 간편 결제도 지원합니다. 온디바이스 AI 기능으로 실시간 통역, 채팅 어시스트, 노트 어시스트 등 다양한 AI 기능을 제공하며, 7년간의 OS 업데이트를 보장합니다. 256GB, 512GB, 1TB의 다양한 저장 용량 옵션을 제공하며, 12GB RAM을 기본 탑재합니다. Wi-Fi 7, 블루투스 5.3, UWB를 지원하며 5G 통신도 지원합니다. 화면 내 지문인식 센서와 얼굴인식을 통한 생체인증을 지원하며, 삼성 녹스를 통한 강력한 보안 기능을 제공합니다.', 'price': 1890000, 'created_at': '2024-01-01'}


In [23]:
# OpenSearch 연결
print("OpenSearch 연결 중...")
client = OpenSearch(
    hosts=[{"host": OPENSEARCH_HOST, "port": OPENSEARCH_PORT}],
    http_auth=(OPENSEARCH_USER, OPENSEARCH_PASSWORD),
    use_ssl=os.getenv('OPENSEARCH_USE_SSL').lower() == 'true',
    verify_certs=os.getenv('OPENSEARCH_VERIFY_CERTS').lower() == 'true',
    ssl_show_warn=False,
)
print(f"[DEBUG] OpenSearch 연결 성공: {client.info()['cluster_name']}")
print(f"[DEBUG] 인덱스 존재 여부: {client.indices.exists(index=OPENSEARCH_INDEX)}")

OpenSearch 연결 중...
[DEBUG] OpenSearch 연결 성공: opensearch-cluster
[DEBUG] 인덱스 존재 여부: True


In [ ]:
# 인덱스 생성 (존재하지 않을 때만 실행됨)
if client.indices.exists(index=OPENSEARCH_INDEX):
    print(f'인덱스 [{OPENSEARCH_INDEX}] 이미 존재 - 생성 스킵')
else:
    client.indices.create(index=OPENSEARCH_INDEX, body={
        'settings': {'index': {'knn': True}},
        'mappings': {'properties': {
            'embedding': {'type': 'knn_vector', 'dimension': 1024},
            'chunk_text': {'type': 'text'},
            'title': {'type': 'text'},
            'category': {'type': 'keyword'}
        }}
    })
    print(f'인덱스 [{OPENSEARCH_INDEX}] 생성 완료')

### 1-2. Split

In [54]:
# 텍스트 청크 분할 실행
print("텍스트 청크 분할 시작...")

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["

", "
", ".", "!", "?", ",", " ", ""]
)

chunks = []
for item in data:
    full_text = f"""제목: {item['title']}
카테고리: {item['category']}
가격: {item['price']}원
설명: {item['description']}"""

    splits = text_splitter.split_text(full_text)
    for split in splits:
        chunks.append({
            "id": item['id'],
            "title": item['title'],
            "category": item['category'],
            "chunk_text": split,
        })

print(f"청크 분할 결과:")
print(f"원본 데이터 수: {len(data)}")
print(f"생성된 청크 수: {len(chunks)}")

print(f"
첫 번째 청크 샘플:")
print("-" * 50)
print(chunks[0]['chunk_text'])
print("-" * 50)
print(f"청크 메타데이터: id={chunks[0]['id']}, title={chunks[0]['title']}, category={chunks[0]['category']}")

텍스트 청크 분할 시작...
청크 분할 결과:
원본 데이터 수: 5
생성된 청크 수: 20

첫 번째 청크 샘플:
--------------------------------------------------
제목: 갤럭시 S24 울트라
카테고리: 스마트폰
가격: 1890000원
--------------------------------------------------
청크 메타데이터: id=1, title=갤럭시 S24 울트라, category=스마트폰


### 1-3. 임베딩 Store

In [55]:
# 임베딩 모델 초기화
print("bge-m3 임베딩 모델 로딩...")
embedder = OllamaEmbeddings(
    model=EMBED_MODEL_NAME
)
print("bge-m3 임베딩 준비 완료")

bge-m3 임베딩 모델 로딩...
bge-m3 임베딩 준비 완료


In [56]:
from dotenv import load_dotenv
load_dotenv(override=True)

# 재로드 후 확인
EMBED_MODEL_NAME = os.getenv('EMBED_MODEL_NAME')
print(EMBED_MODEL_NAME)

bge-m3


```
[문서 임베딩] → [컬렉션 저장] → [인덱싱] → [쿼리 임베딩] → [최근접 이웃 검색] → [Top-K 결과]
```

In [57]:
# OpenSearch 벡터 저장
print("OpenSearch 벡터 저장 시작...")
print(f"처리할 청크 수: {len(chunks)}")

for chunk in tqdm(chunks):
    embedding = embedder.embed_query(chunk['chunk_text'])
    
    client.index(
        index=OPENSEARCH_INDEX,
        body={
            "id": chunk['id'],
            "title": chunk['title'],
            "category": chunk['category'],
            "chunk_text": chunk['chunk_text'],
            "embedding": list(embedding),
        }
    )

print("OpenSearch 벡터 저장 완료!")

# 저장된 벡터 수 확인
client.indices.refresh(index=OPENSEARCH_INDEX)
count = client.count(index=OPENSEARCH_INDEX)['count']
print(f"저장된 벡터 수: {count}")
print(f"검색 설정: Top-{K_RETRIEVAL}")

# 테스트 검색 실행
test_query = "카메라 성능"
test_embedding = embedder.embed_query(test_query)

test_results = client.search(
    index=OPENSEARCH_INDEX,
    body={
        "query": {
            "knn": {
                "embedding": {
                    "vector": test_embedding,
                    "k": K_RETRIEVAL
                }
            }
        }
    }
)

print(f"\n테스트 검색 ('{test_query}'):")
hits = test_results['hits']['hits']
print(f"검색된 문서 수: {len(hits)}")
if hits:
    print(f"첫 번째 결과 (100자): {hits[0]['_source']['chunk_text'][:100]}...")

OpenSearch 벡터 저장 시작...
처리할 청크 수: 20


100%|██████████| 20/20 [00:46<00:00,  2.33s/it]


OpenSearch 벡터 저장 완료!
저장된 벡터 수: 30
검색 설정: Top-4

테스트 검색 ('카메라 성능'):
검색된 문서 수: 10
첫 번째 결과 (100자): 등록일: 2024-01-02...


# 2. 정보 검색 및 텍스트 생성

### 2-1. 모델선언 및 Retrieval

In [58]:
# LLM 모델 선택 및 초기화
# 모델 설정 (gpt/gguf/ollama 중 선택)
# 위 환경설정 셀에서 MODEL_TYPE 조정 가능 ("gpt" 또는 "ollama")
# LLM 모델 선택 및 초기화
TEMPERATURE = 0.2

print(f"{MODEL_TYPE.upper()} 모델 초기화...")


print("Ollama 모델 로딩...")
llm = Ollama(
    model=OLLAMA_MODEL,
    temperature=TEMPERATURE
)
print("Ollama 모델 준비 완료")

print(f"설정된 모델: {MODEL_TYPE.upper()}")
print(f"Temperature: {TEMPERATURE}")

OLLAMA 모델 초기화...
Ollama 모델 로딩...
Ollama 모델 준비 완료
설정된 모델: OLLAMA
Temperature: 0.2


In [59]:
try:
    test_response = llm.invoke("안녕하세요")
    print(f"모델 테스트 성공: {test_response[:50]}...")

except Exception as e:
    print(f"모델 테스트 실패: {e}")
    print("모델 이름을 확인하세요!")

모델 테스트 성공: 안녕하세요! 저는 사용자님의 질문에 답하고 다양한 작업을 돕기 위해 여기에 있습니다. 😊
...


### 2-2. RAG체인 구성 및 Generation

### RAG 체인: 질문 검색 생성, 프롬프트템플릿, 체인연결

```
[사용자 질문] → [쿼리 임베딩] → [Top-K 검색(Chroma)] → [컨텍스트 구성] → [LLM 응답 생성] → [답변 + 출처]
```

In [60]:
# RAG용 프롬프트 템플릿 정의
PROMPT_TEMPLATE = """
다음 컨텍스트를 우선적으로 참고하여 한국어로 간결하고 정확하게 답변하세요.
컨텍스트에 없는 내용은 당신이 알고 있는 지식으로 답변하세요.
가능하면 출처 정보(제목, 카테고리)도 함께 제공하세요.

컨텍스트:
{context}

질문: {question}
답변:
"""
# PromptTemplate 객체 생성
prompt_template = PromptTemplate(
    template=PROMPT_TEMPLATE,
    input_variables=["context", "question"]
)

print("프롬프트 템플릿 생성 완료!")
print("\n템플릿 내용:")
print("-" * 50)
print(PROMPT_TEMPLATE[:200] + "...")
print("-" * 50)

# 템플릿 테스트
test_context = "갤럭시 S24 울트라는 200MP 카메라를 탑재한 스마트폰입니다."
test_question = "갤럭시 S24 울트라의 카메라 성능은?"

formatted_prompt = prompt_template.format(
    context=test_context,
    question=test_question
)

print("\n프롬프트 포맷팅 테스트:")
print(formatted_prompt)

프롬프트 템플릿 생성 완료!

템플릿 내용:
--------------------------------------------------

다음 컨텍스트를 우선적으로 참고하여 한국어로 간결하고 정확하게 답변하세요.
컨텍스트에 없는 내용은 당신이 알고 있는 지식으로 답변하세요.
가능하면 출처 정보(제목, 카테고리)도 함께 제공하세요.

컨텍스트:
{context}

질문: {question}
답변:
...
--------------------------------------------------

프롬프트 포맷팅 테스트:

다음 컨텍스트를 우선적으로 참고하여 한국어로 간결하고 정확하게 답변하세요.
컨텍스트에 없는 내용은 당신이 알고 있는 지식으로 답변하세요.
가능하면 출처 정보(제목, 카테고리)도 함께 제공하세요.

컨텍스트:
갤럭시 S24 울트라는 200MP 카메라를 탑재한 스마트폰입니다.

질문: 갤럭시 S24 울트라의 카메라 성능은?
답변:



In [61]:
vectorstore = OpenSearchVectorSearch(
    opensearch_url=f"https://{OPENSEARCH_HOST}:{OPENSEARCH_PORT}",
    index_name=OPENSEARCH_INDEX,
    embedding_function=embedder,
    http_auth=(OPENSEARCH_USER, OPENSEARCH_PASSWORD),
    vector_field="embedding",
    text_field="chunk_text",
    use_ssl=True,
    verify_certs=False,
    ssl_show_warn=False,
)

retriever = vectorstore.as_retriever(
    search_kwargs={
        "k": K_RETRIEVAL,
        "vector_field": "embedding",
        "text_field": "chunk_text"
    }
)

print("Retriever 생성 완료!")

Retriever 생성 완료!


In [62]:
# 출처 문서 포함 RAG 체인
print("RAG 체인 구성 중 (출처 문서 반환)...")

# 문서 포맷팅 함수
def format_docs(docs):
    """검색된 문서를 텍스트로 변환"""
    formatted = []
    for i, doc in enumerate(docs, 1):
        content = doc.page_content
        title = doc.metadata.get('title', '알 수 없음')
        category = doc.metadata.get('category', '알 수 없음')
        formatted.append(f"[출처 {i}] 제목: {title} | 카테고리: {category}\n{content}")
    return "\n\n".join(formatted)

# Step 1: 검색 및 문맥 준비
retrieval_chain = RunnableParallel(
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough(),
        "source_documents": retriever
    }
)

# Step 2: 답변 생성 체인
answer_chain = (
    {
        "context": lambda x: x["context"],
        "question": lambda x: x["question"]
    }
    | prompt_template
    | llm
    | StrOutputParser()
)

# Step 3: 전체 체인 결합
rag_chain = retrieval_chain | RunnablePassthrough.assign(answer=answer_chain)

print("RAG 체인 생성 완료!")

print("\n체인 구성요소:")
print(f"LLM: {MODEL_TYPE.upper()}")
print(f"Retriever: OpenSearch (K={K_RETRIEVAL})")
print(f"Prompt: 커스텀 템플릿")
print(f"Source Docs: 반환 활성화")

RAG 체인 구성 중 (출처 문서 반환)...
RAG 체인 생성 완료!

체인 구성요소:
LLM: OLLAMA
Retriever: OpenSearch (K=4)
Prompt: 커스텀 템플릿
Source Docs: 반환 활성화


### 질의 테스트

In [51]:
# 질문
question = "카메라 성능이 좋은 스마트폰은?"

print(f"\n질문: {question}")

# 전체 결과 받기
full_result = rag_chain.invoke(question)

# 각 요소 분리 저장
answer = full_result['answer']
source_docs = full_result['source_documents']
context = full_result['context']
question_echo = full_result['question']

# 출력
print(f"\n답변:\n{answer}")

print(f"\n참조 문서 ({len(source_docs)}개):")
for i, doc in enumerate(source_docs, 1):
    title = doc.metadata.get('title', '알 수 없음')
    category = doc.metadata.get('category', '알 수 없음')
    content_preview = doc.page_content[:80].replace('\n', ' ')
    print(f"   [{i}] 제목: {title} | 카테고리: {category}")
    print(f"       {content_preview}...")


질문: 카메라 성능이 좋은 스마트폰은?

답변:
제공된 컨텍스트를 바탕으로 카메라 성능이 좋은 스마트폰 정보를 정리하면 다음과 같습니다.

**답변:**

제공된 컨텍스트에는 **갤럭시 S24 울트라**와 **아이폰 15 프로** 두 스마트폰의 카메라 정보가 포함되어 있습니다.

*   **갤럭시 S24 울트라**
    *   **카메라 시스템:** 200MP 메인 카메라를 포함한 쿼드 카메라 시스템을 탑재하고 있습니다. (200MP 광각, 12MP 초광각, 50MP 3배 망원, 10MP 10배 망원 구성)
    *   **특징:** 갤럭시 AI 기반의 사진 편집 기능(사진 보정, 객체 제거, 배경 변경 등)을 제공합니다.
    *   **출처:** [출처 3] 갤럭시 S24 울트라 | 스마트폰

*   **아이폰 15 프로**
    *   **카메라 시스템:** 48MP 메인, 12MP 울트라와이드, 12MP 3배 망원으로 구성되어 있으며, 프로급 영상 촬영을 위한 Log 비디오 촬영 기능을 지원합니다.
    *   **출처:** [출처 2] 아이폰 15 프로

**출처 정보:**
*   [출처 2] 제목: 아이폰 15 프로 | 카테고리: 스마트폰
*   [출처 3] 제목: 갤럭시 S24 울트라 | 카테고리: 스마트폰

참조 문서 (4개):
   [1] 제목: 갤럭시 S24 울트라 | 카테고리: 스마트폰
       . 삼성 덱스를 지원하여 외부 모니터에 연결하면 데스크톱 환경으로 사용할 수 있으며, 삼성 페이를 통한 간편 결제도 지원합니다. 온디바이스 AI...
   [2] 제목: 아이폰 15 프로 | 카테고리: 스마트폰
       애플 아이폰 15 프로는 2023년 9월에 출시된 애플의 프리미엄 스마트폰으로, 항공우주급 티타늄 소재를 사용하여 가볍고 견고한 디자인을 자랑합...
   [3] 제목: 갤럭시 S24 울트라 | 카테고리: 스마트폰
       삼성 갤럭시 S24 울트라는 삼성전자가 2024년 1월에 출시한 플래그십 스마트폰으로, 2

### 참고. FAST API 연동

In [38]:
!pip install uvicorn==0.20.0
!pip install fastapi pydantic
import uvicorn
from fastapi import FastAPI
from pydantic import BaseModel
from fastapi.middleware.cors import CORSMiddleware

In [39]:
origins = ["*"]

app = FastAPI(title="ML API")

# CORS 미들웨어 추가
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],  # 모든 origin 허용
    allow_credentials=True,
    allow_methods=["GET", "POST", "PUT", "DELETE"],
    allow_headers=["*"],
)

In [40]:
class InDataset(BaseModel):
    question : str

In [41]:
# # 테스트 코드
# # print(x)
# question = "스마트금융과 모집 일정은?"
# response = rag_chain.invoke(question)
# response1 = response["answer"]
# response2 = response["source_documents"]
# print(response1)
# print(response2)

In [42]:
@app.post("/predict", status_code=200)
async def predict(x: InDataset):
    print(x)
    question = x.question
    response = rag_chain.invoke(question)
    response1 = response["answer"]
    response2 = response["source_documents"]
    print(response1)

    return {"prediction": response1,
            "references": [
                {
                    "title": doc.metadata.get("title", ""),
                    "category": doc.metadata.get("category", ""),
                    "content": doc.page_content[:200]
                }
                for doc in response2
            ]}

@app.get("/")
async def root():
    return {"message": "online"}

In [44]:
import nest_asyncio

nest_asyncio.apply()
uvicorn.run(app, host="0.0.0.0", port=9999, log_level="info")